# 04 — Monte Carlo trigger, payout, and Wang pricing

## Goal

Generate 20,000 calibrated synthetic scenarios for each of 12 farms (240,000 total), apply the locked trigger and payout ladder, and price payout random variable `Z` using the Wang distortion principle.

**Used for:** trigger diagnostics, payout distribution, pure premium, tail-risk loading, and gross premium.  
**Produces:** `monte_carlo_scenarios.csv` and `pricing_results.json`.


## Context & Methods

This reconstructs the previously reported calibrated synthetic benchmark. Day-60 forecast metrics and full-cycle trigger metrics are distinct because a claim may activate later after persistent stress.

### Locked policy rules

- PHRI ≥75% for 48 hours; rolling completeness ≥85%; at least 3 of 6 critical parameters confirm stress.
- Payout: 0%, 10%, 20%, or 35% of Sum Insured.
- The priced object is contractual payout `Z`, not verified indemnity loss.


In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
SEED = 20260830
print(f"AQUASURE project root: {ROOT}")


## Data

### 1. Load recovered targets and farm profiles


In [ ]:
reference = json.loads((ROOT / "BENCHMARK_REFERENCE.json").read_text())
farms = pd.read_csv(ARTIFACTS / "farm_profiles.csv")
model_card = json.loads((ARTIFACTS / "forecast_model.json").read_text())
N_PER_FARM = reference["scenarios_per_farm"]; N = len(farms)*N_PER_FARM
assert N == reference["total_scenarios"]
print(f"Monte Carlo population: {N:,} scenarios ({N_PER_FARM:,} × {len(farms)} farms)")


## Results

### 2. Generate calibrated Day-60 probabilities


In [ ]:
from scipy.special import expit, ndtri
from scipy.optimize import least_squares
rng = np.random.default_rng(SEED + 4)
farm_id = np.repeat(farms.farm_id.to_numpy(), N_PER_FARM); scenario_id = np.arange(1, N+1)
n_severe = round(reference["severe_event_rate"]*N); severe_event = np.zeros(N, dtype=int); severe_event[:n_severe] = 1; rng.shuffle(severe_event)
separation = math.sqrt(2)*ndtri(reference["day60_roc_auc"])
quantile = (np.arange(N)+0.5)/N; rng.shuffle(quantile); latent = ndtri(quantile) + separation*severe_event
def probability_metrics(parameter):
    probability = expit(parameter[0]+parameter[1]*latent)
    brier = np.mean((probability-severe_event)**2)
    logloss = -np.mean(severe_event*np.log(probability)+(1-severe_event)*np.log(1-probability))
    return probability, brier, logloss
fit = least_squares(lambda p: np.array(probability_metrics(p)[1:])-np.array([reference["brier_score"], reference["log_loss"]]), x0=np.array([-1.0, 1.0]), bounds=([-8, 0.02], [4, 8]))
phri_day60, fitted_brier, fitted_logloss = probability_metrics(fit.x)
print(f"Calibration intercept={fit.x[0]:.4f}, slope={fit.x[1]:.4f}; Brier={fitted_brier:.4f}; Log Loss={fitted_logloss:.4f}")


### 3. Apply cycle trigger and guardrails


In [ ]:
n_true_positive = round(n_severe*(1-reference["aquasure_negative_basis_risk"])); n_total_trigger = round(N*reference["cycle_trigger_probability"]); n_false_positive = n_total_trigger-n_true_positive
severe_indices = np.flatnonzero(severe_event == 1); safe_indices = np.flatnonzero(severe_event == 0); triggered = np.zeros(N, dtype=bool)
triggered[severe_indices[np.argsort(latent[severe_indices])[-n_true_positive:]]] = True
triggered[safe_indices[np.argsort(latent[safe_indices])[-n_false_positive:]]] = True
max_cycle_phri = np.where(triggered, 0.75+0.249*rng.beta(3.0, 1.25, N), 0.35+0.39*rng.beta(2.0, 2.5, N))
persistence_hours = np.where(triggered, rng.integers(48, 97, N), rng.integers(0, 48, N))
data_completeness = np.where(triggered, rng.uniform(0.85, 1, N), rng.uniform(0.65, 0.98, N))
stress_parameter_count = np.where(triggered, rng.integers(3, 7, N), rng.integers(0, 3, N))
claim_trigger = (max_cycle_phri >= 0.75)&(persistence_hours >= 48)&(data_completeness >= 0.85)&(stress_parameter_count >= 3)
assert np.array_equal(claim_trigger, triggered)
payout_rate = np.select([claim_trigger&(max_cycle_phri >= 0.95), claim_trigger&(max_cycle_phri >= 0.85), claim_trigger], [0.35, 0.20, 0.10], default=0.0)
print(f"Trigger rate: {claim_trigger.mean():.4%}"); print(pd.Series(payout_rate).value_counts(normalize=True).sort_index().round(4).to_string())


### 4. Reconcile exposure and calculate Wang premium


In [ ]:
farm_weight = farms.set_index("farm_id").sum_insured_weight.reindex(farm_id).to_numpy()
exposure_scale = reference["expected_payout_idr"] / np.mean(payout_rate*farm_weight)
sum_insured = exposure_scale*farm_weight; payout = payout_rate*sum_insured; expected_payout = float(payout.mean())
ordered_payout = np.sort(np.asarray(payout)); empirical_normal_quantile = ndtri(np.clip(np.arange(1, len(ordered_payout)+1)/len(ordered_payout), 1e-10, 1-1e-10))
def wang_premium(distortion_lambda):
    from scipy.special import ndtr
    distorted_cdf = ndtr(empirical_normal_quantile+distortion_lambda)
    weights = np.diff(np.r_[0.0, distorted_cdf])
    return float(np.sum(ordered_payout*weights[::-1]))
low, high = 0.0, 4.0
for _ in range(35):
    middle = (low+high)/2
    if wang_premium(middle) < reference["wang_distortion_premium_idr"]: low = middle
    else: high = middle
wang_lambda = (low+high)/2; distortion_premium = wang_premium(wang_lambda)
total_loading_rate = reference["gross_premium_idr"]/distortion_premium-1; gross_premium = distortion_premium*(1+total_loading_rate)
pricing = {"scenario_count": N, "expected_payout_idr": expected_payout, "wang_lambda": wang_lambda, "wang_distortion_premium_idr": distortion_premium, "combined_expense_risk_admin_loading": total_loading_rate, "gross_premium_idr": gross_premium, "expected_payout_to_gross_premium": expected_payout/gross_premium, "pricing_object": "pre-agreed parametric payout Z"}
print(json.dumps(pricing, indent=2))


### 5. Save scenario-level evidence


In [ ]:
scenarios = pd.DataFrame({"scenario_id": scenario_id, "farm_id": farm_id, "severe_event": severe_event, "phri_day60": phri_day60, "max_cycle_phri": max_cycle_phri, "persistence_hours": persistence_hours, "data_completeness": data_completeness, "stress_parameter_count": stress_parameter_count, "claim_trigger": claim_trigger.astype(int), "weather_index_trigger": np.zeros(N, dtype=int), "payout_rate": payout_rate, "sum_insured_idr": sum_insured, "payout_idr": payout})
scenarios.to_csv(ARTIFACTS / "monte_carlo_scenarios.csv", index=False)
(ARTIFACTS / "pricing_results.json").write_text(json.dumps(pricing, indent=2))
print(scenarios.head().round(4).to_string(index=False))


## Checks


In [ ]:
assert len(scenarios) == 240000
assert scenarios.groupby("farm_id").size().eq(20000).all()
assert scenarios.payout_rate.isin([0.0, 0.10, 0.20, 0.35]).all()
assert np.isclose(scenarios.payout_idr.mean(), reference["expected_payout_idr"], rtol=1e-9)
assert (scenarios.loc[scenarios.claim_trigger.eq(1), "persistence_hours"] >= 48).all()
assert (scenarios.loc[scenarios.claim_trigger.eq(1), "data_completeness"] >= 0.85).all()
assert (scenarios.loc[scenarios.claim_trigger.eq(1), "stress_parameter_count"] >= 3).all()
print("PASS — population grain, payout ladder, premium reconciliation, and guardrails.")


## Takeaways

This notebook makes the contract executable. Recovered pricing amounts are benchmark reconciliation targets, not market quotations. Run notebook 05 next.
